# MeMo (Memory Model) with Gemma — Fine-tuning via Keras/KerasHub

This notebook implements the **MeMo (Memory Model)** paper using **Gemma** as the Memory Model.

Paper: https://arxiv.org/abs/2605.15156

The MeMo pipeline:
1. **Fact Extraction** — Extract QA pairs from documents using Gemini
2. **Consolidation** — Merge similar facts
3. **Verification** — Ensure questions are self-contained
4. **Entity Surfacing** — Generate entity-centric questions
5. **Cross-Document Synthesis** — Create multi-hop questions
6. **Fine-tuning** — Train Gemma with LoRA on the generated QA pairs
7. **Inference** — Multi-turn pipeline with Executive (Gemini) + Memory (Gemma)

In [2]:
!pip install -q keras-hub
!pip install -q keras
!pip install -q google-genai datasets

### Configure API Keys

You need:
- **GEMINI_API_KEY** — For the Gemini API (Executive model)
- **HF_TOKEN** — For downloading Gemma from HuggingFace via KerasHub

Add these in Colab's **Secrets** panel.

In [23]:
import os
import json
import ast
from google.colab import userdata
from google import genai
from pydantic import BaseModel

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

client = genai.Client()

### Select Keras Backend

Configure JAX as the backend for best performance. This **must** be set before importing Keras.

In [26]:
# Set Keras backend BEFORE importing keras
os.environ["KERAS_BACKEND"] = "jax"
# Avoid memory fragmentation on JAX backend
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"

import keras
import keras_hub

## Data Pipeline (MeMo Steps 1-5)
### Load HotpotQA Dataset

In [39]:
from datasets import load_dataset

MAX_DOCS = 100
hotpot = load_dataset("hotpotqa/hotpot_qa", 'distractor', split=f"train[:{MAX_DOCS}]")

### Parse Documents

In [40]:
docs = []
sliced_dataset = hotpot.select(range(MAX_DOCS))

for slice in sliced_dataset:
  slice_content = {
      "question": slice['question'],
      "answer": slice['answer'],
      "docs": []
  }
  for title, sentence in zip(slice['context']['title'], slice['context']['sentences']) :
    slice_content["docs"].append(f"title: {title}. sentence: {sentence}")

  docs.append(slice_content)

### QA Generator Helper

In [41]:
class QAPair(BaseModel):
    question: str
    answer: str

class QAList(BaseModel):
    pairs: list[QAPair]

def ask_generator(prompt_text):
    response = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=prompt_text,
        config=genai.types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=QAList,
            temperature=0.3
        )
    )
    try:
        data = ast.literal_eval(response.text)
        return [{"question": item["question"], "answer": item["answer"]} for item in data["pairs"]]
    except:
        return []

### Step 1 & 2 - Fact extraction and Consolidation

In [42]:

qa_extracted = []

for i, doc in enumerate(docs[:1]):
    prompt = f"""
    Given the Wikipedia text below, extract key facts by generating Question and Answer pairs.
    Consolidate similar information into comprehensive questions.
    Text: {doc}
    """
    pairs = ask_generator(prompt)
    for p in pairs: p['doc_id'] = i
    qa_extracted.extend(pairs)

print(qa_extracted)

[{'question': "Between Arthur's Magazine and First for Women, which publication was established first?", 'answer': "Arthur's Magazine was established in 1844, while First for Women was started in 1989.", 'doc_id': 0}, {'question': 'What is the history of Radio City in India?', 'answer': "Radio City is India's first private FM radio station, which began operations in Bengaluru in 2001 and Mumbai in 2004.", 'doc_id': 0}, {'question': 'How did the band Echosmith originate?', 'answer': 'Echosmith was formed in 2009 in Chino, California, and originally operated under the name Ready Set Go! before signing with Warner Bros. Records.', 'doc_id': 0}, {'question': "What is the distinction between Salem College and Wesleyan College regarding women's education in the South?", 'answer': 'Salem College is the oldest female educational institution in the South, while Wesleyan College is the first institution established specifically as a college for women.', 'doc_id': 0}, {'question': 'How did the Fr

### Step 3 - Verification and rewriting

In [43]:
qa_verified = []

for item in qa_extracted:
    prompt = f"""
    Is the following question self-contained? That is, can it be understood without context?
    If not, rewrite the question to be very clear and independent, using the proper nouns from the original text.

    Original Text: {docs[item['doc_id']]}
    Original Question: {item['question']}
    Original Answer: {item['answer']}

    Return only ONE rewritten question (or the original if it's already perfect) and the exact same answer.
    """
    pairs = ask_generator(prompt)
    if pairs:
        qa_verified.append(pairs[0])

print(qa_verified)

[{'question': "Between Arthur's Magazine and First for Women, which publication was established first?", 'answer': "Arthur's Magazine was established in 1844, while First for Women was started in 1989."}, {'question': "Which magazine was started first, Arthur's Magazine or First for Women?", 'answer': "Arthur's Magazine"}, {'question': 'How was the American indie pop band Echosmith formed and what was their original name?', 'answer': 'Echosmith was formed in 2009 in Chino, California, and originally operated under the name Ready Set Go! before signing with Warner Bros. Records.'}, {'question': "What is the distinction between Salem College and Wesleyan College regarding women's education in the Southern United States?", 'answer': 'Salem College is the oldest female educational institution in the South, while Wesleyan College is the first institution established specifically as a college for women.'}, {'question': 'How did the Freeway Complex Fire of 2008 begin?', 'answer': 'The fire be

### Step 4 - Entity Surfacing

In [44]:
qa_entities = []

for i, doc in enumerate(docs):
    prompt = f"""
    Identify the entities named in the Wikipedia text below.
    Generate questions where the answer is EXACTLY the name of the entity.
    Texto: {doc}
    """
    pairs = ask_generator(prompt)
    qa_entities.extend(pairs)

print(qa_entities)

[{'question': "Which company is India's first private FM radio station?", 'answer': 'Radio City'}, {'question': 'Which organization was founded on June 6, 1930?', 'answer': 'Albanian National Team'}, {'question': 'What was the original name of the band Echosmith?', 'answer': 'Ready Set Go!'}, {'question': 'Which institution is the oldest female educational institution in the South?', 'answer': 'Salem College'}, {'question': 'What is the name of the smallest courthouse in the United States that now serves as a museum?', 'answer': 'First Arthur County Courthouse and Jail'}, {'question': 'Which American literary periodical was published in Philadelphia in the 19th century?', 'answer': "Arthur's Magazine"}, {'question': 'Which team defeated HK Kremenchuk in the final of the 2014–15 Ukrainian Hockey Championship?', 'answer': 'ATEK Kiev'}, {'question': "Which woman's magazine is published by Bauer Media Group in the USA?", 'answer': 'First for Women'}, {'question': 'What was the name of the 

### Step 5 - Cross-Document Synthesys

In [45]:
all_texts = "\n".join([f"Doc {i+1}: {d}" for i, d in enumerate(docs)])

prompt_cross = f"""
You have the following Wikipedia articles.
Synthesize logical "Cross-Document" connections by creating difficult questions that require reading BOTH documents to be answered.

Do not mention the documents in the question (e.g., avoid "according to Doc 1").
Articles:
{all_texts}
"""

qa_cross = ask_generator(prompt_cross)
print(qa_cross)

[{'question': "Which magazine was started first, Arthur's Magazine or First for Women?", 'answer': "Arthur's Magazine"}, {'question': 'The Oberoi family is part of a hotel company that has a head office in what city?', 'answer': 'Delhi'}, {'question': 'Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?', 'answer': 'President Richard Nixon'}, {'question': "What nationality was James Henry Miller's wife?", 'answer': 'American'}, {'question': 'Cadmium Chloride is slightly soluble in this chemical, it is also called what?', 'answer': 'alcohol'}, {'question': 'Which tennis player won more Grand Slam titles, Henri Leconte or Jonathan Stark?', 'answer': 'Jonathan Stark'}, {'question': "Which genus of moth in the world's seventh-largest country contains only one species?", 'answer': 'Indogrammodes'}, {'question': 'Who was once considered the best kick boxer in the world, however he has been involved in a number of con

### Combine All QA Pairs

Format the dataset as `prompts` and `responses` for Keras `fit()`.

In [46]:
qa_final = qa_verified + qa_entities + qa_cross

prompts = []
responses = []

for p in qa_final:
    prompts.append(p['question'])
    responses.append(p['answer'])

train_data = {
    "prompts": prompts,
    "responses": responses
}

print(f"Total training examples: {len(prompts)}")
print(f"\nExample:")
print(f"  Prompt:   {prompts[0]}")
print(f"  Response: {responses[0]}")

Total training examples: 966

Example:
  Prompt:   Between Arthur's Magazine and First for Women, which publication was established first?
  Response: Arthur's Magazine was established in 1844, while First for Women was started in 1989.


## Training the MeMo (Memory Model): Gemma via KerasHub

### Load Model

Load Gemma using KerasHub. The model is downloaded from Kaggle automatically.

In [29]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("hf://google/gemma-3-1b-it")
gemma_lm.summary()

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │     999,885,952 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 999,885,952 (3.72 GB)

 Trainable params: 999,885,952 (3.72 GB)

 Non-trainable params: 0 (0.00 B)

### Enable LoRA

Activate LoRA fine-tuning. This freezes the base model weights and adds small trainable matrices, drastically reducing the number of trainable parameters.

In [30]:
# Enable LoRA for the model and set the LoRA rank to 4
gemma_lm.backbone.enable_lora(rank=4)
gemma_lm.summary()

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │   1,000,538,240 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,000,538,240 (3.73 GB)

 Trainable params: 652,288 (2.49 MB)

 Non-trainable params: 999,885,952 (3.72 GB)

### Configure Training

Set up the optimizer and compile the model for fine-tuning.

In [31]:
# Limit the input sequence length to 256 (to control memory usage)
gemma_lm.preprocessor.sequence_length = 256

# Use AdamW (a common optimizer for transformer models)
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
# Exclude layernorm and bias terms from decay
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

### Run Fine-tuning

Train the model on the MeMo-generated QA pairs. This may take a few minutes on a T4 GPU.

In [47]:
gemma_lm.fit(train_data, epochs=1, batch_size=1)

966/966 ━━━━━━━━━━━━━━━━━━━━ 346s 331ms/step - loss: 0.1065 - sparse_categorical_accuracy: 0.2852


## Multi-Turn Inference Pipeline (Executive + Memory)

### Memory Model Inference

Query the fine-tuned Gemma model (Memory) for factual recall.

In [33]:
def query_memory_model(sub_question):
    prompt = f"<start_of_turn>user\n{sub_question}<end_of_turn>\n<start_of_turn>model\n"

    sampler = keras_hub.samplers.TopKSampler(k=5, seed=2)
    gemma_lm.compile(sampler=sampler)

    response = gemma_lm.generate(prompt, max_length=100)

    # Extract only the model's response
    if "<start_of_turn>model\n" in response:
        answer = response.split("<start_of_turn>model\n")[-1]
    else:
        answer = response[len(prompt):]

    # Clean up any remaining special tokens
    answer = answer.replace("<end_of_turn>", "").strip()

    return answer

### Executive Stage 1 — Question Decomposition

The Executive (Gemini) breaks a complex question into atomic sub-questions.

In [34]:
class SubQuestions(BaseModel):
    questions: list[str]

def executive_stage_1_grounding(complex_query):
    print(f"\n[Executive] Breaking down the question: '{complex_query}'")

    prompt = f"""Break down the question into atomic sub-questions.\n
    Question: {complex_query}\n
    Return only the list."""

    response = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=prompt,
        config=genai.types.GenerateContentConfig(response_mime_type="application/json", response_schema=SubQuestions)
    )

    return ast.literal_eval(response.text)["questions"]

### Executive Stage 3 — Synthesis

The Executive (Gemini) synthesizes the final answer from Memory responses.

In [35]:
def executive_stage_3_synthesis(complex_query, memory_responses):
    print("\n[Executive] Synthesizing final answer...")

    context = "\n".join([f"Q: {q}\n A: {a}" for q, a in memory_responses.items()])

    prompt = f"""Using ONLY these memories:
    {context}

    Answer this question: {complex_query}"""

    response = client.models.generate_content(model='gemini-3.1-flash-lite', contents=prompt)

    print(f"""
    ==========================
    [Executive Final Answer]: {response.text}
    ==========================""")

    return response.text

### Full MeMo Inference Pipeline

In [36]:
def memo_inference(complex_query):
    sub_questions = executive_stage_1_grounding(complex_query)
    memory_responses = {}

    for sq in sub_questions:
        print(f"  -> Querying Memory (Gemma): '{sq}'")
        ans = query_memory_model(sq)
        print(f"  <- Memory Responded: '{ans}'")
        memory_responses[sq] = ans

    return executive_stage_3_synthesis(complex_query, memory_responses)

### Testing the Inference with the Original Dataset Question

In [48]:
test_question = docs[0]['question']
expected_answer = docs[0]['answer']

print(f"Dataset Question: {test_question}\n")
print(f"Ground Truth Answer: {expected_answer}\n")

print("--- Starting MEMO Inference ---")
memo_inference(test_question)

Dataset Question: Which magazine was started first Arthur's Magazine or First for Women?

Ground Truth Answer: Arthur's Magazine

--- Starting MEMO Inference ---

[Executive] Breaking down the question: 'Which magazine was started first Arthur's Magazine or First for Women?'
  -> Querying Memory (Gemma): 'When was Arthur's Magazine started?'
  <- Memory Responded: 'Arthur’s Magazine officially started in 1876.'
  -> Querying Memory (Gemma): 'When was First for Women started?'
  <- Memory Responded: 'First for Women started in 1999.'
  -> Querying Memory (Gemma): 'Which start date is earlier?'
  <- Memory Responded: 'Let's analyze the two images to determine which date is earlier:

*   **Image 1:** 1988-03-28
*   **Image 2:** 1988-03-28 

Therefore, **Image 1** is earlier.'

[Executive] Synthesizing final answer...

    [Executive Final Answer]: Arthur's Magazine was started first, as it began in 1876, while First for Women started in 1999.


"Arthur's Magazine was started first, as it began in 1876, while First for Women started in 1999."